# Stage 2 — Elo Ratings & Final Merge

Run the **Stage 1 notebook first**; this notebook reads its intermediate Parquet files.

**What this notebook does**

1. Loads `match_meta.parquet` from Stage 1.
2. Computes each team's pre-match Elo rating for every match:
   ```
   R_new = R_old + K · (S_actual − S_expected)
   S_expected = 1 / (1 + 10^((R_opp − R_team − H) / 400))
   ```
   with `K = 20`, home advantage `H = +100`, initial rating `1500`, in chronological order within each league.
   The pre-match Elo gap is used in Stage 7 to adjust real final results for team strength.
3. Streams the five per-league files from Stage 1 through the merge and appends them to one CSV, so the full
   dataset is never held in memory at once.

**Inputs** — `intermediate/`: `match_meta.parquet`, `stage1_actions_<Country>.parquet`

**Outputs** — `data/`:
- `stage1_stage2_features.csv` — one row per SPADL action with every feature and the per-action Elo columns
- `stage2_elo_matches.csv` — one row per match with pre-match Elo for home and away

## 0. Setup

In [ ]:
!pip install -q tqdm pyarrow

In [ ]:
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
print("pandas:", pd.__version__, "| numpy:", np.__version__)

In [ ]:
# Repository root (the notebooks live one level below it)
PROJECT_ROOT = Path('..') if Path.cwd().name in ('pipeline-stages', 'feature-engineering') else Path('.')
INTERIM_DIR  = PROJECT_ROOT / "intermediate"
OUTPUT_DIR   = PROJECT_ROOT / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEAGUES = {
    "England": "Premier League",
    "Spain":   "La Liga",
    "Italy":   "Serie A",
    "France":  "Ligue 1",
    "Germany": "Bundesliga",
}

# Verify Stage 1 outputs exist.
required = [INTERIM_DIR / "match_meta.parquet"] + \
           [INTERIM_DIR / f"stage1_actions_{c}.parquet" for c in LEAGUES]
for f in required:
    assert f.exists(), f"Missing Stage 1 output: {f}  -- run the Stage 1 notebook first."
print("All Stage 1 outputs present. Ready to proceed.")

## 1. Load match metadata

Small table (~1,826 rows) — loaded fully into memory.

In [ ]:
match_meta = pd.read_parquet(INTERIM_DIR / "match_meta.parquet")
match_meta["dateutc"] = pd.to_datetime(match_meta["dateutc"], errors="coerce")
print(f"Match metadata: {len(match_meta):,} rows")
print("Matches per league:")
print(match_meta.groupby("league_country").size())
match_meta.head(3)

## 2. Elo rating computation (Stage 2 core)

Rating rules:

- Every team starts at **1500**.
- After each match, both teams are updated:
  `R_new = R_old + K · (S_actual − S_expected)`
  with `K = 20`, home advantage `H = +100`.
- Expected score:
  `S_expected = 1 / (1 + 10^((R_opp − R_team − H) / 400))` (H positive for home, negative for away).
- Matches are processed in chronological order per league; ratings are independent across leagues because no fixtures cross league boundaries.

In [ ]:
K_FACTOR       = 20.0
HOME_ADVANTAGE = 100.0
INITIAL_ELO    = 1500.0


def expected_score(r_team: float, r_opp: float, team_is_home: bool) -> float:
    adj = HOME_ADVANTAGE if team_is_home else -HOME_ADVANTAGE
    return 1.0 / (1.0 + 10 ** ((r_opp - r_team - adj) / 400.0))


def compute_elo_for_league(match_subset: pd.DataFrame) -> pd.DataFrame:
    """Walk one league's matches in date order, record pre-match Elo."""
    ms = match_subset.sort_values(["dateutc", "gameweek"],
                                  kind="stable").reset_index(drop=True).copy()

    ratings = defaultdict(lambda: INITIAL_ELO)
    elo_home_pre = np.zeros(len(ms))
    elo_away_pre = np.zeros(len(ms))

    for i, row in ms.iterrows():
        h, a  = int(row["home_team_id"]), int(row["away_team_id"])
        hs, as_ = int(row["home_score"]),  int(row["away_score"])

        r_h, r_a = ratings[h], ratings[a]
        elo_home_pre[i] = r_h
        elo_away_pre[i] = r_a

        # Actual result from home perspective: 1 / 0.5 / 0
        if   hs > as_: s_h = 1.0
        elif hs < as_: s_h = 0.0
        else:          s_h = 0.5
        s_a = 1.0 - s_h

        e_h = expected_score(r_h, r_a, team_is_home=True)
        e_a = expected_score(r_a, r_h, team_is_home=False)

        ratings[h] = r_h + K_FACTOR * (s_h - e_h)
        ratings[a] = r_a + K_FACTOR * (s_a - e_a)

    ms["elo_home_pre"] = elo_home_pre
    ms["elo_away_pre"] = elo_away_pre
    return ms

In [ ]:
elo_parts = []
for country in LEAGUES:
    sub = match_meta[match_meta["league_country"] == country]
    elo_parts.append(compute_elo_for_league(sub))
elo_matches = pd.concat(elo_parts, ignore_index=True)

print(f"Elo computed for {len(elo_matches):,} matches")
print("\nSample (last 3 matches per league):")
elo_matches.sort_values("dateutc").groupby("league_country").tail(3)[
    ["league_country", "dateutc", "home_team_id", "away_team_id",
     "home_score", "away_score", "elo_home_pre", "elo_away_pre"]
].head(15)

### Early-season flag

In the first five matchdays every team starts near 1500, so ratings are still settling. These matches are flagged (`elo_burnin = 1`).

In [ ]:
# gameweek = 0 means the gameweek is unknown in the Wyscout metadata; such matches are not flagged.
elo_matches["elo_burnin"] = (
    (elo_matches["gameweek"] >= 1) & (elo_matches["gameweek"] <= 5)
).astype(int)
print("Burn-in matches per league:")
print(elo_matches.groupby("league_country")["elo_burnin"].sum())

### End-of-season Elo sanity check

Teams that finished higher should have higher Elo. A rough sanity check: top and bottom teams per league by their last recorded rating.

In [ ]:
print("End-of-season Elo leaders & laggards per league:")
for country in LEAGUES:
    sub = elo_matches[elo_matches["league_country"] == country].sort_values("dateutc")
    last = {}
    for _, r in sub.iterrows():
        last[int(r["home_team_id"])] = r["elo_home_pre"]
        last[int(r["away_team_id"])] = r["elo_away_pre"]
    series = pd.Series(last).sort_values(ascending=False)
    print(f"\n  {country} -- top 3:    {series.head(3).round(0).to_dict()}")
    print(  f"  {country} -- bottom 3: {series.tail(3).round(0).to_dict()}")

## 3. Save the per-match Elo table

Stage 4's win-probability model and Stage 6's evaluation both benefit from a standalone per-match Elo lookup. We write it as CSV now.

In [ ]:
elo_out_path = OUTPUT_DIR / "stage2_elo_matches.csv"
elo_matches.to_csv(elo_out_path, index=False)
print(f"Wrote {elo_out_path}  ({elo_out_path.stat().st_size / 1e6:.2f} MB)")

## 4. Streaming merge: Elo features → final CSV

For each of the five leagues we:

1. Load that league's Parquet from Stage 1 (one league at a time).
2. Merge match-level Elo info onto each action.
3. Derive four per-action Elo features from the acting team's perspective:
   - `team_elo_pre` — acting team's pre-match Elo
   - `opp_elo_pre` — opponent's pre-match Elo
   - `elo_diff` = `team_elo_pre − opp_elo_pre`
   - `is_home_team` — 1 if acting team is home, else 0
4. **Append** the rows to the final CSV.
5. Delete the league's DataFrame and garbage-collect before loading the next.

This way we never hold all ~2M actions in RAM at once.

In [ ]:
# Lightweight match-level info to merge onto actions (pre-selected once).
elo_match_info = elo_matches[[
    "game_id", "dateutc", "gameweek",
    "home_team_id", "away_team_id",
    "home_score", "away_score",
    "elo_home_pre", "elo_away_pre", "elo_burnin",
]]

# Column order for the final output. "is_starter" exists in the Stage 1
# parquet as a boolean or nullable boolean.
FINAL_COLUMNS = [
    # identifiers
    "game_id", "league_country", "dateutc", "gameweek",
    "period_id", "time_seconds", "absolute_second", "minute",
    "team_id", "player_id",
    "home_team_id", "away_team_id", "is_home_team",
    # base SPADL
    "action_type", "type_id",
    "result",      "result_id",
    "bodypart",    "bodypart_id",
    "start_x", "start_y", "end_x", "end_y",
    # geometric
    "dx", "dy", "action_distance",
    "dist_to_goal_start", "dist_to_goal_end",
    "angle_to_goal_start", "angle_to_goal_end",
    # rolling window a-1
    "type_id_a1", "result_id_a1", "bodypart_id_a1",
    "start_x_a1", "start_y_a1", "end_x_a1", "end_y_a1",
    "team_id_a1", "player_id_a1", "absolute_second_a1",
    # rolling window a-2
    "type_id_a2", "result_id_a2", "bodypart_id_a2",
    "start_x_a2", "start_y_a2", "end_x_a2", "end_y_a2",
    "team_id_a2", "player_id_a2", "absolute_second_a2",
    # gaps & possession
    "time_since_prev", "dx_from_prev", "dy_from_prev", "dist_from_prev",
    "possession_change",
    # game context
    "home_goals_before", "away_goals_before",
    "goals_for", "goals_against", "goal_difference",
    "score_margin", "score_state",
    "home_score", "away_score",
    # late-game
    "minutes_remaining", "is_post_75",
    "is_starter", "is_substitute",
    "entry_minute", "minutes_on_pitch",
    # Elo (Stage 2 output)
    "elo_home_pre", "elo_away_pre",
    "team_elo_pre", "opp_elo_pre", "elo_diff",
    "elo_burnin",
]

In [ ]:
def merge_elo_onto_actions(actions: pd.DataFrame,
                           elo_info: pd.DataFrame) -> pd.DataFrame:
    """Attach match-level Elo + derived per-action Elo features."""
    df = actions.merge(elo_info, on="game_id", how="left")

    acting_is_home = (df["team_id"].astype(int).values
                      == df["home_team_id"].astype(int).values)
    df["is_home_team"] = acting_is_home.astype(int)
    df["team_elo_pre"] = np.where(acting_is_home, df["elo_home_pre"], df["elo_away_pre"])
    df["opp_elo_pre"]  = np.where(acting_is_home, df["elo_away_pre"], df["elo_home_pre"])
    df["elo_diff"]     = df["team_elo_pre"] - df["opp_elo_pre"]
    return df

In [ ]:
import os

csv_path = OUTPUT_DIR / "stage1_stage2_features.csv"
if csv_path.exists():
    os.remove(csv_path)
print("Ready. Run the merge cell below.")

In [ ]:
import gc

total_rows = 0
total_post75 = 0

for i, country in enumerate(LEAGUES):
    print(f"\n---- {country} ----")

    src = INTERIM_DIR / f"stage1_actions_{country}.parquet"
    actions = pd.read_parquet(src)
    print(f"  Loaded {len(actions):,} actions from {src.name}")

    merged = merge_elo_onto_actions(actions, elo_match_info)
    del actions
    gc.collect()

    keep = [c for c in FINAL_COLUMNS if c in merged.columns]
    missing = [c for c in FINAL_COLUMNS if c not in merged.columns]
    if missing and i == 0:
        print(f"  [warn] missing columns: {missing}")
    merged = merged[keep]

    merged.to_csv(csv_path, mode="a", header=(i == 0), index=False)

    total_rows  += len(merged)
    total_post75 += int(merged["is_post_75"].sum())
    print(f"  Appended {len(merged):,} rows")

    del merged
    gc.collect()

print(f"\n\nAll leagues done. {total_rows:,} total actions.")
print(f"  Post-75\' actions: {total_post75:,} ({total_post75/total_rows*100:.1f}%)")

# Verify the written file
import subprocess
wc = subprocess.run(["wc", "-l", str(csv_path)], capture_output=True, text=True)
line_count = int(wc.stdout.strip().split()[0]) - 1
print(f"\nCSV line count: {line_count:,} (expected {total_rows:,})")
assert line_count == total_rows, f"Line count mismatch! {line_count} vs {total_rows}"
print(f"Saved: {csv_path}  ({csv_path.stat().st_size / 1e6:.1f} MB)")

## 5. Sanity checks on the final output

The final CSV is large (~1-2 GB). We read it **in chunks** and aggregate, so this check itself is RAM-safe.

In [ ]:
import os

# Verify the CSV file size is correct (~335 MB for all 5 leagues).
actual_size = os.path.getsize(csv_path)
print(f"CSV file size on disk: {actual_size / 1e6:.1f} MB")
if actual_size < 100_000_000:
    raise RuntimeError(
        f"CSV is only {actual_size/1e6:.1f} MB — expected ~335 MB. "
        "Re-run the merge cell."
    )

CHUNK_SIZE = 200_000

summary = {"total": 0,
           "post75": 0,
           "by_league": defaultdict(int),
           "by_state": defaultdict(int),
           "by_type_top": defaultdict(int),
           "elo_min": np.inf, "elo_max": -np.inf,
           "null_core": defaultdict(int)}

core_cols = ["game_id", "period_id", "time_seconds", "team_id",
             "action_type", "result", "score_state",
             "team_elo_pre", "opp_elo_pre"]

for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE, low_memory=False):
    summary["total"]  += len(chunk)
    summary["post75"] += int(chunk["is_post_75"].sum())
    for league, n in chunk.groupby("league_country").size().items():
        summary["by_league"][league] += int(n)
    for state, n in chunk["score_state"].value_counts().items():
        summary["by_state"][state] += int(n)
    for a_type, n in chunk["action_type"].value_counts().items():
        summary["by_type_top"][a_type] += int(n)
    summary["elo_min"] = min(summary["elo_min"], chunk["team_elo_pre"].min())
    summary["elo_max"] = max(summary["elo_max"], chunk["team_elo_pre"].max())
    for c in core_cols:
        summary["null_core"][c] += int(chunk[c].isna().sum())

print(f"Total rows:       {summary['total']:,}")
print(f"Post-75' rows:    {summary['post75']:,}  "
      f"({summary['post75']/summary['total']*100:.1f}% -- expected 15-18%)")
print(f"\nRows per league:")
for k, v in summary["by_league"].items():
    print(f"  {k:8s}  {v:,}")
print(f"\nScore-state distribution:")
total = summary["total"]
for k, v in summary["by_state"].items():
    print(f"  {k:12s}  {v:,}  ({v/total*100:.1f}%)")
print(f"\nTop 10 action types:")
for a_type, n in sorted(summary["by_type_top"].items(), key=lambda x: -x[1])[:10]:
    print(f"  {a_type:18s}  {n:,}")
print(f"\nElo range: {summary['elo_min']:.0f} - {summary['elo_max']:.0f}")
print(f"\nNull counts in core columns:")
for k, v in summary["null_core"].items():
    print(f"  {k:18s}  {v}")

### Goal-reconstruction check

The goals reconstructed from our action stream should match Wyscout's official final scores. A mismatch would mean we mis-classified shots / own-goals during Wyscout → SPADL conversion.

We only need score-relevant rows for this check (shots that were successes, plus all own-goals), so loading just those three columns keeps it RAM-safe.

In [ ]:
import os

# Load only the minimal columns we need for the reconstruction check.
check_cols = ["game_id", "team_id", "action_type", "result"]
check_df = pd.read_csv(csv_path, usecols=check_cols, low_memory=False)

SHOT_TYPES = {"shot", "shot_penalty", "shot_freekick"}

# Goals from shots: acting team scores.
scored_by_shots = (
    check_df[check_df["action_type"].isin(list(SHOT_TYPES))
             & (check_df["result"] == "success")]
    .groupby(["game_id", "team_id"]).size()
    .rename("rec_goals").reset_index()
)

# Own goals: the OPPOSITE team gets the credit.
own_goals = check_df[check_df["result"] == "owngoal"].merge(
    match_meta[["game_id", "home_team_id", "away_team_id"]], on="game_id"
)
own_goals["beneficiary"] = np.where(
    own_goals["team_id"] == own_goals["home_team_id"],
    own_goals["away_team_id"], own_goals["home_team_id"],
)
own_goals_agg = (
    own_goals.groupby(["game_id", "beneficiary"]).size()
    .rename("rec_goals").reset_index()
    .rename(columns={"beneficiary": "team_id"})
)

reconstructed = pd.concat([scored_by_shots, own_goals_agg], ignore_index=True)
reconstructed = reconstructed.groupby(["game_id", "team_id"],
                                      as_index=False)["rec_goals"].sum()

# Official: long form of (game_id, team_id, final_score)
official = pd.concat([
    match_meta[["game_id", "home_team_id", "home_score"]]
        .rename(columns={"home_team_id": "team_id", "home_score": "off_goals"}),
    match_meta[["game_id", "away_team_id", "away_score"]]
        .rename(columns={"away_team_id": "team_id", "away_score": "off_goals"}),
], ignore_index=True)

check = official.merge(reconstructed, on=["game_id", "team_id"], how="left").fillna(0)
check["rec_goals"] = check["rec_goals"].astype(int)
check["off_goals"] = check["off_goals"].astype(int)
check["match"]     = check["rec_goals"] == check["off_goals"]

agree = check["match"].mean() * 100
print(f"Reconstructed team-goal counts agree with official scores on "
      f"{agree:.1f}% of team-match rows.")
if agree < 99:
    print("\nLargest mismatches (first 10):")
    print(check[~check["match"]].head(10))

---

## Done

Outputs in `data/`: `stage1_stage2_features.csv` (one row per action) and `stage2_elo_matches.csv`
(pre-match Elo per match).

**Next → Stage 3** (action values) and **Stage 4** (in-game stakes), both of which read `stage1_stage2_features.csv`.